# LAB 3 — External Volume Setup

This notebook performs the one-time Lab 3 infrastructure setup. Shared parameters and paths are imported from `lab03_config`.

## 1. Load shared configuration

In [0]:
%run ./lab03_config

## 2. Create or confirm the external Unity Catalog volume

In [0]:
spark.sql(
    f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS
    `{catalog}`.`{schema}`.`{volume_name}`
    LOCATION '{external_storage_location}'
    COMMENT 'Lab 3 streaming and incremental ingestion files in ADLS'
    """
)

volume_details = spark.sql(
    f"DESCRIBE VOLUME "
    f"`{catalog}`.`{schema}`.`{volume_name}`"
)
display(volume_details)

## 3. Validate that the existing volume uses the expected ADLS path

In [0]:
details = {
    row["key"]: row["value"]
    for row in volume_details.collect()
    if "key" in row.asDict() and "value" in row.asDict()
}

# DESCRIBE VOLUME output can differ slightly by runtime,
# so the displayed result remains the primary visual check.
print(f"Expected location: {external_storage_location}")
print("Confirm that volume_type is EXTERNAL and storage_location matches.")

## 4. Create the Lab 3 directory structure

In [0]:
required_paths = [
    source_path,
    landing_path,
    staging_initial_path,
    staging_evolved_path,
    staging_renamed_path,
    staging_malformed_path,
    autoloader_schema_path,
    autoloader_checkpoint_path,
    eventhub_bronze_checkpoint_path,
    eventhub_silver_checkpoint_path,
]

for path in required_paths:
    dbutils.fs.mkdirs(path)
    dbutils.fs.ls(path)

    if create_folder_markers:
        dbutils.fs.put(
            f"{path}/_READY",
            "Lab 3 directory initialized.",
            overwrite=True
        )

    print(f"Ready: {path}")

## 5. Display setup configuration

In [0]:
configuration = [
    ("Catalog", catalog),
    ("Schema", schema),
    ("Volume", volume_name),
    ("Storage account", storage_account),
    ("ADLS container", container),
    ("External storage location", external_storage_location),
    ("Volume root", volume_root),
    ("Source path", source_path),
    ("Expected source file", source_file_path),
    ("Landing path", landing_path),
    ("Initial staging", staging_initial_path),
    ("Evolved staging", staging_evolved_path),
    ("Renamed staging", staging_renamed_path),
    ("Malformed staging", staging_malformed_path),
    ("Auto Loader schema", autoloader_schema_path),
    ("Auto Loader checkpoint", autoloader_checkpoint_path),
    ("Event Hub Bronze checkpoint", eventhub_bronze_checkpoint_path),
    ("Event Hub Silver checkpoint", eventhub_silver_checkpoint_path),
    ("File Bronze table", file_bronze_table),
    ("File Silver table", file_silver_table),
    ("Event Hub Bronze table", eventhub_bronze_table),
    ("Event Hub Silver table", eventhub_silver_table),
]

display(
    spark.createDataFrame(
        configuration,
        ["parameter", "value"]
    )
)

## 6. Verify folders

In [0]:
display(dbutils.fs.ls(volume_root))

In [0]:
display(dbutils.fs.ls(staging_root))

## 7. Final validation

In [0]:
assert external_storage_location.startswith("abfss://")
assert volume_root.startswith(
    f"/Volumes/{catalog}/{schema}/"
)
assert source_path != landing_path
assert autoloader_schema_path != autoloader_checkpoint_path

for path in required_paths:
    dbutils.fs.ls(path)

print("Lab 3 setup completed successfully.")
print(f"Upload or confirm the source file at: {source_file_path}")
print("Next notebook: lab03_01_file_generation")